# Notebook 06 — Importancia de Atributos y Análisis de Adjetivos

## Objetivo

Interpretar qué términos lingüísticos asocia el mejor modelo lineal con fake vs real, y validar hipótesis sobre adjetivos.

> Los coeficientes reflejan patrones del dataset, no veracidad factual.

In [22]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.paths import *
from nlp.plotting import setup_style, save_figure

setup_style()

import joblib
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from nlp.io import load_splits
from nlp.modeling import get_linear_feature_weights

## 1. Cargar mejor modelo lineal (politics)

In [23]:
TEXT_COL = 'clean_full_text_without_stopwords'
pol_train, _, pol_test = load_splits('politics', columns=['label', TEXT_COL])

model_path = RESULTS_MODELS / 'best_baseline_politics.joblib'
config_path = RESULTS_MODELS / 'best_baseline_politics_config.json'

if model_path.exists():
    pipe = joblib.load(model_path)
    clf = pipe.named_steps['clf']
    if hasattr(clf, 'coef_') or hasattr(clf, 'feature_log_prob_'):
        print('Modelo baseline cargado desde results/models/best_baseline_politics.joblib')
    else:
        pipe = None
else:
    pipe = None

if pipe is None:
    print('Entrenando fallback TF-IDF + LR para análisis de coeficientes...')
    X_train = pol_train[TEXT_COL].fillna('')
    pipe = Pipeline([
        ('vec', TfidfVectorizer(ngram_range=(1, 2), max_features=30000, min_df=2)),
        ('clf', LogisticRegression(C=1, max_iter=1000, random_state=RANDOM_STATE)),
    ])
    pipe.fit(X_train, pol_train['label'])

if config_path.exists():
    print('Config guardada:', pd.read_json(config_path, typ='series')[['model', 'vectorizer', 'text_field']].to_dict())


Modelo lineal entrenado para análisis de coeficientes.


## 2. Coeficientes más importantes

In [24]:

feature_names = pipe.named_steps['vec'].get_feature_names_out()
coefs = get_linear_feature_weights(pipe)
importance = pd.DataFrame({'term': feature_names, 'coefficient': coefs})
importance['abs_coef'] = importance['coefficient'].abs()
importance = importance.sort_values('abs_coef', ascending=False)

top_fake = importance.nlargest(30, 'coefficient')[['term', 'coefficient']]
top_real = importance.nsmallest(30, 'coefficient')[['term', 'coefficient']]

feature_importance = pd.concat([
    top_fake.assign(direction='fake'),
    top_real.assign(direction='real'),
])
feature_importance.to_csv(RESULTS_METRICS / 'feature_importance.csv', index=False)
feature_importance.head(10)


,term,coefficient,direction
28374,video,9.151223,fake
28317,via,6.432082,fake
12001,hillary,5.925492,fake
20721,read,4.085574,fake
11150,gop,4.001007,fake
28137,us,3.741528,fake
16562,mr,3.645880,fake
28862,watch,3.294748,fake
28133,url,3.022919,fake
1696,america,2.937081,fake


In [25]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, title, color in [
    (axes[0], top_fake.head(15), 'Términos asociados a FAKE', '#e74c3c'),
    (axes[1], top_real.head(15).assign(coefficient=lambda x: x['coefficient'].abs()), 'Términos asociados a REAL', '#2ecc71'),
]:
    sns.barplot(data=data.iloc[::-1], y='term', x='coefficient', color=color, ax=ax)
    ax.set_title(title)
save_figure(fig, RESULTS_FIGURES / 'feature_importance_top_terms.png')
plt.show()


## 3. Términos frecuentes y n-gramas por clase

Buscamos expresiones sensacionalistas/emocionales (fake) vs estilo periodístico formal (real, ej. marcas Reuters).

In [26]:

def top_ngrams_by_class(df, col, label, n=20):
    texts = df[df['label'] == label][col].fillna('').astype(str)
    vec = CountVectorizer(ngram_range=(2, 2), token_pattern=r'(?u)\\b\\w+\\b')
    matrix = vec.fit_transform(texts)
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    features = vec.get_feature_names_out()
    ranked = sorted(zip(features, counts, strict=False), key=lambda x: x[1], reverse=True)
    return ranked[:n]

fake_bi = top_ngrams_by_class(pol_train, TEXT_COL, 1)
real_bi = top_ngrams_by_class(pol_train, TEXT_COL, 0)
print('Bigramas frecuentes FAKE:', fake_bi[:10])
print('Bigramas frecuentes REAL:', real_bi[:10])


Bigramas frecuentes FAKE: [('hillary clinton', 1656), ('donald trump', 1630), ('white house', 1135), ('united states', 1083), ('twitter com', 781), ('pic twitter', 775), ('new york', 773), ('barack obama', 572), ('state department', 545), ('fox news', 539)]
Bigramas frecuentes REAL: [('donald trump', 5755), ('white house', 5545), ('united states', 5400), ('washington reuters', 3878), ('new york', 3416), ('barack obama', 2530), ('president barack', 2299), ('hillary clinton', 2190), ('trump said', 2184), ('president donald', 1949)]


## 4. Análisis de adjetivos con spaCy

In [27]:

import spacy
import subprocess
import sys

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
    nlp = spacy.load('en_core_web_sm')

def extract_adjectives(texts, sample_size=3000):
    adj_counter = Counter()
    for doc in nlp.pipe(texts.head(sample_size).astype(str), batch_size=256):
        for token in doc:
            if token.pos_ == 'ADJ':
                adj_counter[token.lemma_.lower()] += 1
    return adj_counter

fake_adj = extract_adjectives(pol_train[pol_train['label']==1][TEXT_COL])
real_adj = extract_adjectives(pol_train[pol_train['label']==0][TEXT_COL])

adj_rows = []
for adj, cnt in fake_adj.most_common(50):
    adj_rows.append({'adjective': adj, 'count': cnt, 'class': 'fake'})
for adj, cnt in real_adj.most_common(50):
    adj_rows.append({'adjective': adj, 'count': cnt, 'class': 'real'})

adj_df = pd.DataFrame(adj_rows)
adj_df.to_csv(RESULTS_METRICS / 'adjectives_by_class.csv', index=False)
adj_df.head(10)


,adjective,count,class
0,black,1475,fake
1,many,1006,fake
2,last,987,fake
3,american,970,fake
4,new,932,fake
5,presidential,820,fake
6,good,789,fake
7,political,785,fake
8,first,763,fake
9,former,750,fake


In [28]:

# Gráfico comparativo de adjetivos
fake_top = pd.DataFrame(fake_adj.most_common(15), columns=['adjective', 'fake_count'])
real_top = pd.DataFrame(real_adj.most_common(15), columns=['adjective', 'real_count'])
adj_plot = fake_top.merge(real_top, on='adjective', how='outer').fillna(0)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(adj_plot))
width = 0.35
ax.barh(x - width/2, adj_plot['fake_count'], width, label='Fake', color='#e74c3c')
ax.barh(x + width/2, adj_plot['real_count'], width, label='Real', color='#2ecc71')
ax.set_yticks(x)
ax.set_yticklabels(adj_plot['adjective'])
ax.set_xlabel('Frecuencia')
ax.set_title('Adjetivos más frecuentes por clase')
ax.legend()
save_figure(fig, RESULTS_FIGURES / 'adjectives_by_class.png')
plt.show()


## Conclusiones

- Los términos con coeficiente positivo se asocian estadísticamente con fake en este dataset.
- Los adjetivos pueden diferir en tono emocional/sensacionalista vs formal.
- Estas asociaciones son específicas del corpus y no implican verificación factual.